In [ ]:
import pymssql
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score

# data_split: training, calibration and test splits

For each representation, dataset A (individual items) and dataset B (summed scores), this notebook
writes a training, a calibration and a test split.

1. Load the cleaned data from `data/raw_data/`: `sx_clean_df.csv` (dataset A) and
   `sx_clean_Total_df.csv` (dataset B).
2. Remove PhqTen, scale each individual questionnaire item to 0 to 1 by its maximum possible score
   (dataset A), convert the one-hot columns to integers, and rescale fq_anx_dep by its six-item
   maximum (dataset B).
3. Split 1: a stratified 80/20 build-test split, which gives the test split.
4. Split 2: a stratified 80/20 train-calibration split of the build data, which gives the training
   and calibration splits, 64 and 16 percent of patients overall.
5. Save the splits to `data/train/`, `data/cal/` and `data/test/`, and the patient indices of each
   split to `split_indices_indv_seed42.pkl` and `split_indices_sum_seed42.pkl`.

Both representations use seed 42 and hold the same patients in each split.

# LOAD CLEANED DATAFRAMES

In [ ]:
DATA_SEED = 42

## INDIVIDUAL MHQ SCORES DATA

In [ ]:
data_X_mhq_df_3 = pd.read_csv('data/raw_data/sx_clean_df.csv')
data_X_mhq_df_2 = data_X_mhq_df_3.drop(data_X_mhq_df_3.columns[0], axis=1)
data_X_mhq_df = data_X_mhq_df_2.drop(data_X_mhq_df_2.columns[0], axis=1)
print(f'shape for individual scores df: {data_X_mhq_df.shape}\n')
data_X_mhq_df.info()

### Drop PhqTen

In [ ]:
# --- drop PhqTen ---
def drop_phqten(df):
    """Return a copy of df with the PhqTen column removed. No-op if absent."""
    if "PhqTen" not in df.columns:
        print("PhqTen not present, nothing dropped")
        return df.copy()
    out = df.drop(columns=["PhqTen"])
    print(f"Dropped PhqTen: {df.shape} -> {out.shape}")
    return out

data_X_mhq_df = drop_phqten(data_X_mhq_df)
assert "PhqTen" not in data_X_mhq_df.columns

### Scale individual MHQ items to 0-1

In [ ]:
# --- scale individual MHQ items to 0-1 ---
# Each questionnaire item is divided by its maximum possible score. The scales start at
# 0, so PHQ, with four response options, has a maximum of 3. The summed scores follow the
# same rule (PHQ divides by 27, 9 items times 3), so the two representations stay
# comparable. The divisor is a fixed property of the instrument and independent of the
# data, so the split that follows is unaffected.

ITEM_MAX = {"phq": 3, "gad": 3, "sias": 4, "pdss": 4, "fq": 8}
NON_MHQ  = ("gender", "education", "alcoholintake", "currentmedication",
            "treatmenttype", "oh_")

def item_family(col):
    """Return the questionnaire family for a column, or None if it is not an MHQ item."""
    c = col.lower()
    if c.startswith(NON_MHQ):
        return None
    for fam in ITEM_MAX:
        if c.startswith(fam):
            return fam
    return None

def scale_mhq_items(df, item_max=ITEM_MAX):
    """
    Divide every MHQ item column by its maximum possible score, mapping it onto 0-1.

    Demographic, one-hot and target columns are left untouched. Raises if a column
    holds a value above its instrument maximum, below zero, or missing, so a coding
    error cannot pass through silently.
    """
    out  = df.copy()
    cols = {c: item_family(c) for c in df.columns}
    mhq  = {c: f for c, f in cols.items() if f is not None}
    skipped = [c for c, f in cols.items() if f is None]

    # idempotence guard: re-running the cell must not divide twice
    already = [c for c in mhq if df[c].max() <= 1.0]
    if len(already) == len(mhq) and len(mhq) > 0:
        print("All MHQ items already sit within 0-1. Nothing rescaled.")
        return out

    audit = []
    for c, fam in mhq.items():
        mx = item_max[fam]
        lo, hi, na = df[c].min(), df[c].max(), df[c].isna().sum()
        if na:
            raise ValueError(f"{c}: {na} missing values. Fill or drop before scaling.")
        if lo < 0:
            raise ValueError(f"{c}: minimum {lo} is below 0.")
        if hi > mx:
            raise ValueError(f"{c}: maximum {hi} exceeds the {fam} item maximum of {mx}.")
        out[c] = df[c] / mx
        audit.append({"column": c, "family": fam, "divisor": mx,
                      "raw_min": lo, "raw_max": hi,
                      "scaled_min": round(out[c].min(), 4),
                      "scaled_max": round(out[c].max(), 4)})

    audit = pd.DataFrame(audit)
    print(f"Scaled {len(audit)} MHQ item columns, left {len(skipped)} columns untouched.")
    print("\nDivisor used per family, with the resulting range:")
    print(audit.groupby(["family", "divisor"])
               .agg(n_columns=("column", "size"),
                    raw_max=("raw_max", "max"),
                    scaled_min=("scaled_min", "min"),
                    scaled_max=("scaled_max", "max"))
               .to_string())
    print(f"\nUntouched: {skipped}")

    check = out[list(mhq)]
    assert check.min().min() >= 0 and check.max().max() <= 1, \
        "An MHQ column fell outside 0-1 after scaling."
    print(f"\nAll MHQ columns now within 0 to 1. "
          f"Full frame range: {out.drop(columns=['TreatmentType']).min().min():.3f} "
          f"to {out.drop(columns=['TreatmentType']).max().max():.3f}")
    return out

data_X_mhq_df = scale_mhq_items(data_X_mhq_df)
data_X_mhq_df.head(5)

### View data

In [ ]:
data_X_mhq_df.info()

In [ ]:
data_X_mhq_df.head(5)

## TOTAL MHQ DATA

In [ ]:
data_X_sum_df_3 = pd.read_csv('data/raw_data/sx_clean_Total_df.csv')
data_X_sum_df_2 = data_X_sum_df_3.drop(data_X_sum_df_3.columns[0], axis=1)
data_X_sum_df = data_X_sum_df_2.drop(data_X_sum_df_2.columns[0], axis=1)
print(f'shape for sum scores df: {data_X_sum_df.shape}\n')
data_X_sum_df.info()

### Drop PhqTen

In [ ]:
# --- drop PhqTen ---
data_X_sum_df = drop_phqten(data_X_sum_df)
assert "PhqTen" not in data_X_sum_df.columns

### View data

In [ ]:
data_X_sum_df.info()

In [ ]:
data_X_sum_df.head(5)

## Convert one-hot columns from boolean to integer 0/1

In [ ]:
# --- convert one-hot columns from boolean to integer 0/1 ---
# The oh_* columns arrive as pandas bool. Converting them to int before the split gives
# every saved CSV the values 0 and 1, so every later step reads one dtype. The block
# check raises when a one-hot block does not sum to exactly 1 per patient, the mark of an
# encoding fault.

ONE_HOT_PREFIX = "oh_"

def one_hot_to_int(df, label, prefix=ONE_HOT_PREFIX):
    """
    Convert the one-hot columns to integer 0 and 1 and validate the encoding.

    Accepts columns stored as bool, as 0/1 numbers, or as the strings "True" and
    "False", which is what a CSV round trip can produce, so the cell is safe to
    re-run. Raises on a missing value, on any value that is not 0 or 1, and on a
    one-hot block that does not sum to exactly 1 for every patient.
    """
    out  = df.copy()
    cols = [c for c in df.columns if c.startswith(prefix)]
    if not cols:
        print(f"{label}: no columns starting with '{prefix}'. Nothing converted.")
        return out

    before = {c: str(df[c].dtype) for c in cols}
    for c in cols:
        s = df[c]
        if s.isna().any():
            raise ValueError(f"{label}: {c} has {int(s.isna().sum())} missing values.")
        if pd.api.types.is_bool_dtype(s):
            out[c] = s.astype(int)
        elif pd.api.types.is_numeric_dtype(s):
            bad = set(pd.unique(s)) - {0, 1, 0.0, 1.0}
            if bad:
                raise ValueError(f"{label}: {c} holds values other than 0 and 1: {sorted(bad)[:5]}")
            out[c] = s.astype(int)
        else:
            mapped = s.astype(str).str.strip().str.lower().map(
                {"true": 1, "false": 0, "1": 1, "0": 0})
            if mapped.isna().any():
                raise ValueError(f"{label}: {c} holds text that is not True or False.")
            out[c] = mapped.astype(int)

    blocks = {}
    for c in cols:
        blocks.setdefault(c.rsplit("_", 1)[0], []).append(c)
    for name, members in blocks.items():
        row_sum = out[members].sum(axis=1)
        if not (row_sum == 1).all():
            raise ValueError(f"{label}: block '{name}' does not sum to 1 for every patient. "
                             f"Row-sum counts: {row_sum.value_counts().sort_index().to_dict()}")

    print(f"{label}: converted {len(cols)} one-hot columns from "
          f"{sorted(set(before.values()))} to int.")
    for name, members in blocks.items():
        print(f"   {name:12s} {len(members)} columns, every row sums to 1, "
              f"positives per column: {[int(out[m].sum()) for m in members]}")
    assert out[cols].isin([0, 1]).all().all(), f"{label}: a one-hot value is not 0 or 1."
    print(f"   dtypes now: {sorted({str(out[c].dtype) for c in cols})}")
    return out

data_X_mhq_df = one_hot_to_int(data_X_mhq_df, "individual")
data_X_sum_df = one_hot_to_int(data_X_sum_df, "summed")

In [ ]:
# --- correct the fq_anx_dep divisor ---
# fq_anx_dep sums six FQ items, each scored 0 to 8, so its maximum possible score is 48.
# The cleaned data divides it by 40, the five-item maximum of fq_ago, fq_bld and fq_soc,
# which puts seven patients above 1.0. The stored value is an integer sum over 40, so
# multiplying by 40 recovers the sum exactly before the division by 48.

FQ_ANX_DEP_OLD_DIVISOR = 40  # five items x 8, the divisor in the cleaned data
FQ_ANX_DEP_NEW_DIVISOR = 48  # six items x 8, the maximum possible score

def fix_fq_anx_dep(df, label, col="fq_anx_dep",
                   old_div=FQ_ANX_DEP_OLD_DIVISOR, new_div=FQ_ANX_DEP_NEW_DIVISOR):
    """
    Rescale fq_anx_dep from the five-item divisor to the six-item one, in place on a copy.

    Recovers the raw item sum by multiplying by the old divisor, checks that it is a
    whole number within 0 and the new divisor, then divides by the new one. Detects
    which divisor the column currently carries by testing which multiplier returns
    integers, so the cell is safe to re-run and will not rescale twice.
    """
    out = df.copy()
    if col not in out.columns:
        print(f"{label}: '{col}' not present. Nothing changed.")
        return out

    s = out[col]
    if s.isna().any():
        raise ValueError(f"{label}: {col} has {int(s.isna().sum())} missing values.")

    raw_old = s * old_div
    raw_new = s * new_div
    dev_old = float(np.abs(raw_old - raw_old.round()).max())
    dev_new = float(np.abs(raw_new - raw_new.round()).max())
    TOL = 1e-6

    if dev_new < TOL and dev_old > TOL:
        print(f"{label}: '{col}' already carries the /{new_div} divisor "
              f"(max {s.max():.4f}). Nothing changed.")
        return out
    if not (dev_old < TOL and dev_new > TOL):
        raise ValueError(
            f"{label}: cannot tell which divisor '{col}' currently uses (deviation from "
            f"integer: /{old_div} -> {dev_old:.2e}, /{new_div} -> {dev_new:.2e}). "
            f"Inspect the column before rescaling.")

    raw = raw_old.round()
    if raw.min() < 0 or raw.max() > new_div:
        raise ValueError(f"{label}: recovered raw sums span {raw.min():.0f} to "
                         f"{raw.max():.0f}, outside 0 to {new_div}.")
    out[col] = raw / new_div

    print(f"{label}: '{col}' rescaled from /{old_div} to /{new_div}.")
    print(f"   recovered raw sum: {raw.min():.0f} to {raw.max():.0f} "
          f"(six items x 8 = {new_div} maximum possible)")
    print(f"   before: min {s.min():.4f}  max {s.max():.4f}  mean {s.mean():.4f}  "
          f"above 1: {int((s > 1).sum())}")
    print(f"   after : min {out[col].min():.4f}  max {out[col].max():.4f}  "
          f"mean {out[col].mean():.4f}  above 1: {int((out[col] > 1).sum())}")
    assert out[col].between(0, 1).all(), f"{label}: {col} still outside 0 to 1."
    return out

data_X_sum_df = fix_fq_anx_dep(data_X_sum_df, "summed")

### View data

In [ ]:
data_X_mhq_df.head(5)

In [ ]:
data_X_sum_df.head(5)

# SPLIT 1 - build-test (80/20)

## version A - INDV.

In [ ]:
# set X-Y data: INDIVIDUAL SCORES - version A
xA_data = data_X_mhq_df[data_X_mhq_df.columns[~data_X_mhq_df.columns.str.startswith('Treatment')]]
yA_data = data_X_mhq_df['TreatmentType']

print(f'INDIVIDUAL SCORES')
print(f"Original data size = {xA_data.shape} \n")

# Build-test split 
xA_build, xA_test, yA_build, yA_test = train_test_split(
    xA_data, 
    yA_data, 
    test_size = 0.20, 
    stratify = yA_data,
    random_state = DATA_SEED)

print(f"Original Y value counts  = ")
yA_data.value_counts()

### print split shapes 

In [ ]:
print(f"Build dataframe size = {xA_build.shape}")
print(f"Build Y value counts size = {yA_build.shape}, => ")
yA_build.value_counts()

In [ ]:
print(f"TEST dataframe size  = {xA_test.shape}")
print(f"TEST Y value counts size = {xA_test.shape}, => ")
yA_test.value_counts()

### SAVE version A

In [ ]:
xA_build.to_csv('data/build_data/x_indv_build.csv', index=True)

xA_test.to_csv('data/test/x_indv_test.csv', index=True)

yA_build.to_csv('data/build_data/y_indv_build.csv', index=True)

yA_test.to_csv('data/test/y_indv_test.csv', index=True)

print(f"Build (for next split) data with individual scores saved as 'x_indv_build.csv' and 'y_indv_build.csv'")
print(f"TEST data with individual scores saved as 'x_indv_test.csv', 'y_indv_test.csv'")


### View Split data 

In [ ]:
xA_build.info()

In [ ]:
xA_test.info()

## version B - SUM

In [ ]:
# set X-Y data: SUMMED SCORES - version B
xB_data = data_X_sum_df[data_X_sum_df.columns[~data_X_sum_df.columns.str.startswith('Treatment')]]
yB_data = data_X_sum_df['TreatmentType']

print(f'SUM SCORES')
print(f"Original data size = {xB_data.shape} \n")


# Build-test split 
xB_build, xB_test, yB_build, yB_test = train_test_split(
    xB_data, 
    yB_data, 
    test_size = 0.20, 
    stratify = yB_data,
    random_state = DATA_SEED)

print(f"Original Y value counts  = ")
yB_data.value_counts()

### print split shapes 

In [ ]:
print(f"Build dataframe size = {xB_build.shape}")
print(f"Build Y value counts  = ")
yB_build.value_counts()

In [ ]:
print(f"TEST dataframe size  = {xB_test.shape}")
print(f"TEST Y value counts  = ")
yB_test.value_counts()

### SAVE version B

In [ ]:
xB_build.to_csv('data/build_data/x_sum_build.csv', index=True)

xB_test.to_csv('data/test/x_sum_test.csv', index=True)

yB_build.to_csv('data/build_data/y_sum_build.csv', index=True)

yB_test.to_csv('data/test/y_sum_test.csv', index=True)

print(f"Build (for next split) data with individual scores saved as 'x_sum_build.csv', 'y_sum_build.csv' ")
print(f"TEST data with individual scores saved as 'x_sum_test.csv', 'y_sum_test.csv'")


### View Split data 

In [ ]:
xB_build.info()

In [ ]:
xB_test.info()

# SPLIT 2 - train-calibrate (80/20) <- from build (80%)

In [ ]:
data_X_indv_build_df_2 = pd.read_csv('data/build_data/x_indv_build.csv')
data_X_indv_build_df = data_X_indv_build_df_2.drop(data_X_indv_build_df_2.columns[0], axis=1)

print(f'shape for SUM df: {data_X_indv_build_df.shape}\n')
data_X_indv_build_df.info()


In [ ]:
data_Y_indv_build_df_2 = pd.read_csv('data/build_data/y_indv_build.csv')
data_Y_indv_build_df = data_Y_indv_build_df_2.drop(data_Y_indv_build_df_2.columns[0], axis=1)

print(f'shape for SUM df: {data_Y_indv_build_df.shape}\n')
data_Y_indv_build_df.info()


In [ ]:
data_Y_indv_build_df.value_counts()

In [ ]:
data_X_sum_build_df_2 = pd.read_csv('data/build_data/x_sum_build.csv')
data_X_sum_build_df = data_X_sum_build_df_2.drop(data_X_sum_build_df_2.columns[0], axis=1)

print(f'shape for SUM df: {data_X_sum_build_df.shape}\n')
data_X_sum_build_df.info()

In [ ]:
data_Y_sum_build_df_2 = pd.read_csv('data/build_data/y_sum_build.csv')
data_Y_sum_build_df = data_Y_sum_build_df_2.drop(data_Y_sum_build_df_2.columns[0], axis=1)

print(f'shape for SUM df: {data_Y_sum_build_df.shape}\n')
data_Y_sum_build_df.info()

In [ ]:
data_Y_sum_build_df.value_counts()

## version A - INDV.

In [ ]:
# set X-Y data: INDIVIDUAL SCORES - version A
xA_data = xA_build
yA_data = yA_build

print(f'INDIVIDUAL SCORES')
print(f"Original data size = {xA_data.shape} \n")

# Train-calibration split
xA_train, xA_cal, yA_train, yA_cal = train_test_split(
    xA_data, 
    yA_data, 
    test_size = 0.20
    , 
    stratify = yA_data,
    random_state = DATA_SEED)

print(f"Original Y value counts  = ")
yA_data.value_counts()

### print split shapes

In [ ]:
print(f"TRAIN dataframe size = {xA_train.shape}")
print(f"TRAIN Y value counts size = {yA_train.shape}, => ")
yA_train.value_counts()

In [ ]:
print(f"CALIBRATION dataframe size  = {xA_cal.shape}")
print(f"CALIBRATION Y value counts size = {yA_cal.shape}, => ")
yA_cal.value_counts()

### SAVE version A

In [ ]:
xA_train.to_csv('data/train/x_indv_train.csv', index=True)
print(f"TRAIN data with individual scores saved as 'x_indv_train.csv'")

xA_cal.to_csv('data/cal/x_indv_calibration.csv', index=True)
print(f"CALIBRATION data with individual scores saved as 'x_indv_calibration.csv'")

yA_train.to_csv('data/train/y_indv_train.csv', index=True)
print(f"TRAIN data with individual scores saved as 'y_indv_train.csv'")

yA_cal.to_csv('data/cal/y_indv_calibration.csv', index=True)
print(f"CALIBRATION data with individual scores saved as 'y_indv_calibration.csv'")

### View Split data

In [ ]:
xA_train.info()

In [ ]:
xA_cal.info()

## version B - SUM

In [ ]:
# set X-Y data: SUMMED SCORES - version B
xB_data = xB_build
yB_data = yB_build

print(f'SUM SCORES')
print(f"Original data size = {xB_data.shape} \n")

# Train-calibration split
xB_train, xB_cal, yB_train, yB_cal = train_test_split(
    xB_data, 
    yB_data, 
    test_size = 0.20, 
    stratify = yB_data,
    random_state = DATA_SEED)

print(f"Original Y value counts  = ")
yB_data.value_counts()


### print split shapes

In [ ]:
print(f"TRAIN dataframe size = {xB_train.shape}")
print(f"TRAIN Y value counts  = ")
yB_train.value_counts()

In [ ]:
print(f"CALIBRATION dataframe size  = {xB_cal.shape}")
print(f"CALIBRATION Y value counts  = ")
yB_cal.value_counts()

### SAVE version B

In [ ]:
xB_train.to_csv('data/train/x_sum_train.csv', index=True)
print(f"TRAIN data with sum scores saved as 'x_sum_train.csv'")

xB_cal.to_csv('data/cal/x_sum_calibration.csv', index=True)
print(f"CALIBRATION data with sum scores saved as 'x_sum_calibration.csv'")

yB_train.to_csv('data/train/y_sum_train.csv', index=True)
print(f"TRAIN data with sum scores saved as 'y_sum_train.csv'")

yB_cal.to_csv('data/cal/y_sum_calibration.csv', index=True)
print(f"CALIBRATION data with sum scores saved as 'y_sum_calibration.csv'")

### View Split data

In [ ]:
xB_train.info()

In [ ]:
xB_cal.info()

# FINAL SPLITS

In [ ]:
import pickle
split_indices_indv = {
    "train": xA_train.index.to_numpy(),
    "cal":   xA_cal.index.to_numpy(),
    "test":  xA_test.index.to_numpy(),
}
split_indices_sum = {
    "train": xB_train.index.to_numpy(),
    "cal":   xB_cal.index.to_numpy(),
    "test":  xB_test.index.to_numpy(),
}

# guarantee 1: same patients in each split across the two versions
for k in ("train", "cal", "test"):
    assert np.array_equal(np.sort(split_indices_indv[k]),
                          np.sort(split_indices_sum[k])), \
        f"{k}: individual and sum splits reference different rows!"

# guarantee 2: splits are disjoint and cover all 2525 patients
allidx = np.concatenate(list(split_indices_sum.values()))
assert len(allidx) == 2525 and len(np.unique(allidx)) == 2525, \
    "splits overlap or miss patients!"

with open(f"split_indices_indv_seed{DATA_SEED}.pkl", "wb") as f:
    pickle.dump(split_indices_indv, f)
with open(f"split_indices_sum_seed{DATA_SEED}.pkl", "wb") as f:
    pickle.dump(split_indices_sum, f)

print("Saved split_indices for both versions.")
print(f"  train={len(split_indices_sum['train'])}  "
      f"cal={len(split_indices_sum['cal'])}  test={len(split_indices_sum['test'])}")
print("  individual and sum reference identical patients")

## INDIVIDUAL MHQ

In [ ]:
print(f"-->> TEST data with individual scores saved as 'x_indv_test.csv'")
print(f"-->> TEST data with individual scores saved as 'y_indv_test.csv'")
print(f"TEST data size             = {xA_test.shape}\n\n")

print(f"-->> Build (for next split) data with individual scores saved as 'x_indv_build.csv'")
print(f"-->> Build (for next split) data with individual scores saved as 'y_indv_build.csv'")
print(f"Build data size            = {xA_build.shape}\n\n")

print(f"-->> TRAIN data with individual scores saved as 'x_indv_train.csv'")
print(f"-->> TRAIN data with individual scores saved as 'y_indv_train.csv'")
print(f"TRAIN data size            = {xA_train.shape}\n\n")

print(f"-->> CALIBRATION data with individual scores saved as 'x_indv_calibration.csv'")
print(f"-->> CALIBRATION data with individual scores saved as 'y_indv_calibration.csv'")
print(f"CALIBRATION data size      = {xA_cal.shape}")

In [ ]:
print('TEST Y value counts:')
yA_test.value_counts()

In [ ]:
print('Build Y value counts:')
yA_build.value_counts()

In [ ]:
print('TRAIN Y value counts:')
yA_train.value_counts()

In [ ]:
print('CALIBRATION Y value counts:')
yA_cal.value_counts()

## TOTAL/SUM MHQ

In [ ]:
print(f"-->> TEST data with sum scores saved as 'x_sum_test.csv'")
print(f"-->> TEST data with sum scores saved as 'y_sum_test.csv'")
print(f"TEST data size             = {xB_test.shape}\n\n")

print(f"-->> Build (for next split) data with sum scores saved as 'x_sum_build.csv'")
print(f"-->> Build (for next split) data with sum scores saved as 'y_sum_build.csv'")
print(f"Build data size            = {xB_build.shape}\n\n")

print(f"-->> TRAIN data with sum scores saved as 'x_sum_train.csv'")
print(f"-->> TRAIN data with sum scores saved as 'y_sum_train.csv'")
print(f"TRAIN data size            = {xB_train.shape}\n\n")

print(f"-->> CALIBRATION data with sum scores saved as 'x_sum_calibration.csv'")
print(f"-->> CALIBRATION data with sum scores saved as 'y_sum_calibration.csv'")
print(f"CALIBRATION data size      = {xB_cal.shape}")

In [ ]:
print('TEST Y value counts:')
yB_test.value_counts()

In [ ]:
print('Build Y value counts:')
yB_build.value_counts()

In [ ]:
print('TRAIN Y value counts:')
yB_train.value_counts()

In [ ]:
print('CALIBRATION Y value counts:')
yB_cal.value_counts()